In [1]:
import time
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score,roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
import xgboost as xgb

In [2]:
X,y = make_classification(n_samples = 5000, n_features = 20, n_informative = 12, n_redundant = 4, weights = [0.8, 0.2], random_state = 42)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state = 42)

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
N_ITER = 30
SCORING = "roc_auc"

In [5]:
param_grids = {
    "Random Forest": {
        "model": RandomForestClassifier(random_state=42, n_jobs=-1),
        "params": {
            "n_estimators": [100, 200, 300],
            "max_depth": [None, 10, 20, 30],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2", 0.5, 0.8],
        },
    },
    "SklearnGBC": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
            "max_depth": [2, 3, 4, 6],
            "subsample": [0.6, 0.8, 1.0],
            "max_features": ["sqrt", 0.5, 0.8, 1.0],
        },
    },
    "XGBoost": {
        "model": xgb.XGBClassifier(
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
        ),
        "params": {
            "n_estimators": [100, 200, 300],
            "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
            "max_depth": [2, 3, 4, 6],
            "subsample": [0.6, 0.8, 1.0],
            "colsample_bytree": [0.5, 0.8, 1.0],
            "reg_alpha": [0.0, 0.01, 0.1, 1.0],
            "reg_lambda": [0.1, 1.0, 5.0, 10.0],
        },
    },
}

In [6]:
summary_records = []
reports = {}

In [7]:
for name, config in param_grids.items():
    start_time = time.time()

    search = RandomizedSearchCV(
        estimator=config["model"],
        param_distributions=config["params"],
        n_iter=N_ITER,
        scoring=SCORING,
        cv=cv,
        random_state=42,
        n_jobs=-1 if name == "SklearnGBC" else 1,
    )
    search.fit(X_train, y_train)
    elapsed = time.time() - start_time

    best_clf = search.best_estimator_
    y_pred = best_clf.predict(X_test)
    y_prob = best_clf.predict_proba(X_test)[:, 1]

    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=["Class 0 (Majority)", "Class 1 (Minority)"],
        digits=4,
        output_dict=True,
    )
    report_text = classification_report(
        y_test,
        y_pred,
        target_names=["Class 0 (Majority)", "Class 1 (Minority)"],
        digits=4,
    )
    reports[name] = report_text

    summary_records.append(
        {
            "Model": name,
            "CV Best ROC-AUC": round(search.best_score_, 4),
            "Test ROC-AUC": round(roc_auc_score(y_test, y_prob), 4),
            "Minority Recall": round(
                report_dict["Class 1 (Minority)"]["recall"], 4
            ),
            "Minority Precision": round(
                report_dict["Class 1 (Minority)"]["precision"], 4
            ),
            "Minority F1": round(
                report_dict["Class 1 (Minority)"]["f1-score"], 4
            ),
            "Macro F1": round(report_dict["macro avg"]["f1-score"], 4),
            "Search Time (s)": round(elapsed, 2),
        }
    )

In [9]:
print("BENCHMARK SUMMARY TABLE")
print("-" * 80)
summary_df = pd.DataFrame(summary_records).sort_values(
    by="Test ROC-AUC", ascending=False
)
print(summary_df.to_string(index=False))

print("\n" + "-" * 80)
print("DETAILED CLASSIFICATION REPORTS PER MODEL")
print("=" * 80)
for name, rep in reports.items():
    print(f"\n--- {name} ---")
    print(rep)

BENCHMARK SUMMARY TABLE
--------------------------------------------------------------------------------
        Model  CV Best ROC-AUC  Test ROC-AUC  Minority Recall  Minority Precision  Minority F1  Macro F1  Search Time (s)
      XGBoost           0.9735        0.9781           0.8522              0.9402       0.8941    0.9343            64.38
   SklearnGBC           0.9712        0.9770           0.7931              0.9527       0.8656    0.9174           107.57
Random Forest           0.9675        0.9716           0.7635              0.9627       0.8516    0.9093            80.61

--------------------------------------------------------------------------------
DETAILED CLASSIFICATION REPORTS PER MODEL

--- Random Forest ---
                    precision    recall  f1-score   support

Class 0 (Majority)     0.9428    0.9925    0.9670       797
Class 1 (Minority)     0.9627    0.7635    0.8516       203

          accuracy                         0.9460      1000
         macro avg